# ESC-50 demo: Mel-CNN vs frozen Wav2Vec 2.0 (linear and attention heads)
Run the cells in order. Needs a T4 GPU, the `ESC-50` folder, the `cache/` folder and the `checkpoints/` folder.

## 1. Setup

In [ ]:
%cd /content/esc50-wav2vec2
!git pull
!ls checkpoints cache results

In [ ]:
import glob, json, random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchaudio
from IPython.display import Audio, display
from transformers import Wav2Vec2Model

from data.esc50 import SAMPLE_RATE, load_waveform
from train import build_model
from utils.seed import set_seed

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT, LAYER = "ESC-50", 12

meta = pd.read_csv(f"{ROOT}/meta/esc50.csv")
class_names = meta.drop_duplicates("target").sort_values("target")["category"].tolist()
test_meta = meta[meta.fold == 5].reset_index(drop=True)   # held-out test fold
print(device, "| test clips:", len(test_meta))

In [ ]:
CKPTS = {
    "Model 1: Mel-CNN":            ("mel_cnn",    "checkpoints/mel_cnn_seed42.pt"),
    "Model 2: Wav2Vec + linear":   ("w2v_linear", f"checkpoints/w2v_linear_L{LAYER}_seed42.pt"),
    "Model 3: Wav2Vec + attention": ("w2v_attn",   f"checkpoints/w2v_attn_L{LAYER}_seed42.pt"),
}
models = {}
for label, (name, path) in CKPTS.items():
    if Path(path).exists():
        m = build_model(name)
        m.load_state_dict(torch.load(path, map_location=device))
        models[label] = (name, m.to(device).eval())
    else:
        print("missing checkpoint (skipped):", path)

backbone = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device).eval()
for p in backbone.parameters():
    p.requires_grad = False          # frozen

mel_tf = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=1024,
                                              hop_length=320, n_mels=64).to(device)
to_db = torchaudio.transforms.AmplitudeToDB(top_db=80).to(device)
print("loaded:", list(models))

## 2. Live parameter inspection

In [ ]:
for label, (name, m) in models.items():
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{label}: total {total:,} | trainable {trainable:,}")
bb_total = sum(p.numel() for p in backbone.parameters())
bb_train = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f"Wav2Vec 2.0 backbone: total {bb_total:,} | trainable {bb_train:,}  (frozen)")

if "Model 3: Wav2Vec + attention" in models:
    print()
    print(models["Model 3: Wav2Vec + attention"][1])

## 3. Results table (test fold 5)

In [ ]:
rows = [json.load(open(f)) for f in sorted(glob.glob("results/*.json"))
        if not Path(f).name.startswith("eval_")]
cols = ["model", "layer", "seed", "trainable_params", "best_val_acc", "test_acc", "test_macro_f1"]
display(pd.DataFrame(rows).reindex(columns=cols))

## 4. Loss and accuracy curves (TensorBoard)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 5. Live predictions on held-out test clips

In [ ]:
@torch.no_grad()
def predict(wave, top_k=3):
    w = wave.to(device).unsqueeze(0)                                  # (1, 80000)
    mel = to_db(mel_tf(w)).unsqueeze(1)                               # (1, 1, 64, 251)
    wn = (w - w.mean(1, keepdim=True)) / torch.sqrt(w.var(1, keepdim=True) + 1e-7)
    feats = backbone(wn, output_hidden_states=True).hidden_states[LAYER]   # (1, 249, 768)
    out, attn = {}, None
    for label, (name, m) in models.items():
        if name == "mel_cnn":
            logits = m(mel)
        elif name == "w2v_attn":
            logits, attn = m(feats, return_attention=True)
        else:
            logits = m(feats)
        top = torch.softmax(logits, 1)[0].topk(top_k)
        out[label] = [(class_names[i], float(v)) for v, i in zip(top.values, top.indices)]
    return out, (attn[0].cpu().numpy() if attn is not None else None)


def show_clip(i):
    row = test_meta.iloc[i]
    wave = load_waveform(Path(ROOT) / "audio" / row.filename)
    true = class_names[row.target]
    display(Audio(wave.numpy(), rate=SAMPLE_RATE))
    out, attn = predict(wave)
    print(f"TRUE: {true}   ({row.filename})")
    for label, preds in out.items():
        tag = "[correct]" if preds[0][0] == true else "[wrong]  "
        print(f"  {tag} {label}: " + ", ".join(f"{n} {p:.0%}" for n, p in preds))
    if attn is not None:
        t = np.linspace(0, 5, len(attn))
        fig, ax = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
        ax[0].plot(np.linspace(0, 5, len(wave)), wave.numpy(), lw=0.5)
        ax[0].set_ylabel("waveform")
        ax[1].plot(t, attn)
        ax[1].fill_between(t, attn, alpha=0.3)
        ax[1].set_ylabel("attention weight")
        ax[1].set_xlabel("seconds")
        plt.show()
    print("-" * 70)

random.seed(0)
for i in random.sample(range(len(test_meta)), 3):
    show_clip(i)

Pick any clip yourself by changing the index (0 to 399):

In [ ]:
show_clip(10)

## 6. Live training run (Wav2Vec linear head, ~1 minute)
Uses a different seed so the saved results are not overwritten.

In [ ]:
!python train.py --model w2v_linear --layer 12 --seed 7

## 7. Full test-fold evaluation (accuracy, macro F1, ROC-AUC, confusion matrix)

In [ ]:
!python evaluate.py --model mel_cnn --root ESC-50
!python evaluate.py --model w2v_attn --root ESC-50 --layer 12

## 8. Optional: try your own audio
Upload a .wav file to Colab and set the path.

In [ ]:
# custom = "my_sound.wav"
# wave = load_waveform(Path(custom))
# out, attn = predict(wave); print(out)
# display(Audio(wave.numpy(), rate=SAMPLE_RATE))